In [4]:
import pandas as pd
import os
import gcamreader
from params import (
    CAR_ANNUAL_SERVICE,
    BUS_ANNUAL_SERVICE,
    TRUCK_ANNUAL_SERVICE,
)

# Source

* Data Source:
    * December 2020 Motor Vehicle Registration Statistics, Ministry of Land, Infrastructure and Transport (MOLIT). (`../resources/VRS-2020-MOLIT`, `../resources/VRS-2025-MOLIT`)

* Implemented Input Files
    * `/input/policy/korea-2035/transportation/ZEV_calibrate.xml`

We calibrate ZEV service output using Korea’s 2020 ZEV enrollment data. The number of registered vehicles is converted into performance units based on the assumptions specified in `basic_assumption.ipynb`. For passenger cars, calibration in the model is further disaggregated into the `Car` and `Large Car and Truck` categories. Detailed implementation steps are provided below.

In [5]:
car_veh_zev_2020 = 117616
bus_veh_zev_2020 = 1837
truck_veh_zev_2020 = 15436
car_veh_fcev_2020 = 10517
bus_veh_fcev_2020 = 75
truck_veh_fcev_2020 = 0
car_veh_zev_2025 = 713835
bus_veh_zev_2025 = 14779
truck_veh_zev_2025 = 169744
car_veh_fcev_2025 = 35941
bus_veh_fcev_2025 = 1539
truck_veh_fcev_2025 = 49

In [6]:
UNIT_CONVERTER = 1e6

In [7]:
car_output_zev_2020 = car_veh_zev_2020 * CAR_ANNUAL_SERVICE / UNIT_CONVERTER
bus_output_zev_2020 = bus_veh_zev_2020 * BUS_ANNUAL_SERVICE / UNIT_CONVERTER
truck_output_zev_2020 = truck_veh_zev_2020 * TRUCK_ANNUAL_SERVICE / UNIT_CONVERTER

car_output_fcev_2020 = car_veh_fcev_2020 * CAR_ANNUAL_SERVICE / UNIT_CONVERTER
bus_output_fcev_2020 = bus_veh_fcev_2020 * BUS_ANNUAL_SERVICE / UNIT_CONVERTER
truck_output_fcev_2020 = truck_veh_fcev_2020 * TRUCK_ANNUAL_SERVICE / UNIT_CONVERTER

car_output_zev_2025 = car_veh_zev_2025 * CAR_ANNUAL_SERVICE / UNIT_CONVERTER
bus_output_zev_2025 = bus_veh_zev_2025 * BUS_ANNUAL_SERVICE / UNIT_CONVERTER
truck_output_zev_2025 = truck_veh_zev_2025 * TRUCK_ANNUAL_SERVICE / UNIT_CONVERTER

car_output_fcev_2025 = car_veh_fcev_2025 * CAR_ANNUAL_SERVICE / UNIT_CONVERTER
bus_output_fcev_2025 = bus_veh_fcev_2025 * BUS_ANNUAL_SERVICE / UNIT_CONVERTER
truck_output_fcev_2025 = truck_veh_fcev_2025 * TRUCK_ANNUAL_SERVICE / UNIT_CONVERTER

In [8]:
dbpath = "../../output/"
dbfile = "database_basexdb"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', '..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Reference


In [9]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Reference']

In [10]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [11]:
i = 159
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=['Reference'], regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

transport service output by tech


,Units,scenario,region,sector,subsector,technology,Year,value
0,million pass-km,Reference,South Korea,trn_aviation_intl,International Aviation,BEV,2035,4.188620e-03
1,million pass-km,Reference,South Korea,trn_aviation_intl,International Aviation,BEV,2040,3.478500e-02
2,million pass-km,Reference,South Korea,trn_aviation_intl,International Aviation,BEV,2045,2.693970e-01
3,million pass-km,Reference,South Korea,trn_aviation_intl,International Aviation,BEV,2050,1.900350e+00
4,million pass-km,Reference,South Korea,trn_aviation_intl,International Aviation,BEV,2055,3.531660e+00
...,...,...,...,...,...,...,...,...
901,million ton-km,Reference,South Korea,trn_shipping_intl,International Ship,Liquids,2080,1.643780e+06
902,million ton-km,Reference,South Korea,trn_shipping_intl,International Ship,Liquids,2085,1.693100e+06
903,million ton-km,Reference,South Korea,trn_shipping_intl,International Ship,Liquids,2090,1.742860e+06
904,million ton-km,Reference,South Korea,trn_shipping_intl,International Ship,Liquids,2095,1.789120e+06


In [12]:
df[(df['Year'] == 2025) & (df['sector'].isin(['trn_pass_road_LDV_4W'])) & (df['technology'] == 'BEV')]

,Units,scenario,region,sector,subsector,technology,Year,value
405,million pass-km,Reference,South Korea,trn_pass_road_LDV_4W,Car,BEV,2025,12033.30
496,million pass-km,Reference,South Korea,trn_pass_road_LDV_4W,Large Car and Truck,BEV,2025,5741.57


In [16]:
from utils import records_to_gcam_trn_fixed_output_xml

UNIT_CONVERTER = 1e6  # million pass-km or million ton-km

# ---- 2020 outputs (million units)
car_output_zev_2020   = car_veh_zev_2020   * CAR_ANNUAL_SERVICE   / UNIT_CONVERTER
bus_output_zev_2020   = bus_veh_zev_2020   * BUS_ANNUAL_SERVICE   / UNIT_CONVERTER
truck_output_zev_2020 = truck_veh_zev_2020 * TRUCK_ANNUAL_SERVICE / UNIT_CONVERTER

car_output_fcev_2020   = car_veh_fcev_2020   * CAR_ANNUAL_SERVICE   / UNIT_CONVERTER
bus_output_fcev_2020   = bus_veh_fcev_2020   * BUS_ANNUAL_SERVICE   / UNIT_CONVERTER
truck_output_fcev_2020 = truck_veh_fcev_2020 * TRUCK_ANNUAL_SERVICE / UNIT_CONVERTER


# ---- LDV split (set these to your intended shares)
SHARE_CAR = 12033.30 / (12033.30+5741.57)
SHARE_LARGE = 1 - SHARE_CAR

car_zev_2020   = car_output_zev_2020 * SHARE_CAR
large_zev_2020 = car_output_zev_2020 * SHARE_LARGE

car_fcev_2020   = car_output_fcev_2020 * SHARE_CAR
large_fcev_2020 = car_output_fcev_2020 * SHARE_LARGE

car_zev_2025   = car_output_zev_2025 * SHARE_CAR
large_zev_2025 = car_output_zev_2025 * SHARE_LARGE

car_fcev_2025   = car_output_fcev_2025 * SHARE_CAR
large_fcev_2025 = car_output_fcev_2025 * SHARE_LARGE


In [20]:
records = [
    # freight road - medium truck
    dict(supplysector="trn_freight_road", subsector="Medium truck", tech="BEV",
         year=2020, fixedOutput=truck_output_zev_2020),
    dict(supplysector="trn_freight_road", subsector="Medium truck", tech="FCEV",
         year=2020, fixedOutput=truck_output_fcev_2020),

    # passenger road - bus
    dict(supplysector="trn_pass_road", subsector="Bus", tech="BEV",
         year=2020, fixedOutput=bus_output_zev_2020),
    dict(supplysector="trn_pass_road", subsector="Bus", tech="FCEV",
         year=2020, fixedOutput=bus_output_fcev_2020),

    # LDV 4W - car
    dict(supplysector="trn_pass_road_LDV_4W", subsector="Car", tech="BEV",
         year=2020, fixedOutput=car_zev_2020),
    dict(supplysector="trn_pass_road_LDV_4W", subsector="Car", tech="FCEV",
         year=2020, fixedOutput=car_fcev_2020),

    # LDV 4W - large car & truck
    dict(supplysector="trn_pass_road_LDV_4W", subsector="Large Car and Truck", tech="BEV",
         year=2020, fixedOutput=large_zev_2020),
    dict(supplysector="trn_pass_road_LDV_4W", subsector="Large Car and Truck", tech="FCEV",
         year=2020, fixedOutput=large_fcev_2020),

    dict(supplysector="trn_freight_road", subsector="Medium truck", tech="BEV",
         year=2025, fixedOutput=truck_output_zev_2025),
    dict(supplysector="trn_freight_road", subsector="Medium truck", tech="FCEV",
         year=2025, fixedOutput=truck_output_fcev_2025),

    # passenger road - bus
    dict(supplysector="trn_pass_road", subsector="Bus", tech="BEV",
         year=2025, fixedOutput=bus_output_zev_2025),
    dict(supplysector="trn_pass_road", subsector="Bus", tech="FCEV",
         year=2025, fixedOutput=bus_output_fcev_2025),

    # LDV 4W - car
    dict(supplysector="trn_pass_road_LDV_4W", subsector="Car", tech="BEV",
         year=2025, fixedOutput=car_zev_2025),
    dict(supplysector="trn_pass_road_LDV_4W", subsector="Car", tech="FCEV",
         year=2025, fixedOutput=car_fcev_2025),

    # LDV 4W - large car & truck
    dict(supplysector="trn_pass_road_LDV_4W", subsector="Large Car and Truck", tech="BEV",
         year=2025, fixedOutput=large_zev_2025),
    dict(supplysector="trn_pass_road_LDV_4W", subsector="Large Car and Truck", tech="FCEV",
         year=2025, fixedOutput=large_fcev_2025),
]

out_path = "../../input/policy/korea-2035/transportation/ZEV_calibrate.xml"
records_to_gcam_trn_fixed_output_xml(records, out_path, region_name="South Korea")

print("Wrote:", out_path)

Wrote: ../../input/policy/korea-2035/transportation/ZEV_calibrate.xml
